In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import requests, zipfile, io, os
import pandas as pd

In [ ]:
# Listings fetch
domain = "datasets.techmatrix.it/airml"
token = "DI_xeno_2026"

cities = ["sicilia", "trentino", "venezia", "roma", "puglia",
          "napoli", "firenze", "milano", "bergamo", "bologna"]

for city in cities:
    data_dir = os.path.join("./data", city)
    if os.path.exists(data_dir):
        print(f"Skipping {city}: folder already exists")
        continue
    
    url = f"https://{domain}/listings/{city}.zip?token={token}"
    resp = requests.get(url, stream=True)
    if resp.ok:
        os.makedirs(data_dir, exist_ok=True)
        zip_path = os.path.join("./data", f"{city}.zip")
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(8192):
                if chunk:
                    f.write(chunk)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(path=data_dir)
        os.remove(zip_path)
    else:
        print(f"Failed to download {city}: {resp.status_code}")

In [ ]:
# # Reviews fetch

# for city in cities:
#     url = f"https://{domain}/reviews/{city}.zip?token={token}"
#     resp = requests.get(url, stream=True)
#     if resp.ok:
#         data_dir = os.path.join("./data", city)
#         if not os.path.exists(data_dir):
#             os.makedirs(data_dir, exist_ok=True)
#         zip_path = os.path.join("./data", f"{city}.zip")
#         with open(zip_path, "wb") as f:
#             for chunk in resp.iter_content(8192):
#                 if chunk:
#                     f.write(chunk)
#         with zipfile.ZipFile(zip_path, "r") as z:
#             z.extractall(path=data_dir)
#         os.remove(zip_path)
#     else:
#         print(f"Failed to download {city}: {resp.status_code}")

# Filtering

In [ ]:
COLS_TO_DROP = {
    # URL / immagini
    "listing_url", "picture_url", "host_thumbnail_url", "host_picture_url", "host_url",
    # Testuali / identificativi listing
    "name", "description", "neighborhood_overview", "calendar_updated",
    # Identificatori di scraping / metadati tecnici
    "scrape_id", "last_scraped", "source",
    # Identificatori personali / dati host
    "host_id", "host_name", "host_since", "host_location", "host_about",
    "host_neighbourhood", "host_listings_count", "host_total_listings_count",
    "host_verifications", "host_has_profile_pic", "host_identity_verified",
    # Metriche risposta host
    "host_response_time", "host_response_rate", "host_acceptance_rate", "host_is_superhost",
    # Location duplicate / non predittive
    "neighbourhood", "neighbourhood_group_cleansed",
    # Testo derivabile / calcolato
    "bathrooms_text", "first_review", "last_review",
    # Calcolati host (aggregati)
    "calculated_host_listings_count", "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms", "calculated_host_listings_count_shared_rooms",
}
dfs = []

for sub in os.listdir("./data"):
    subpath = os.path.join("./data", sub)
    if not os.path.isdir(subpath):
        continue
    csv_path = os.path.join(subpath, "listings.csv")
    if os.path.exists(csv_path):
        dfs.append(pd.read_csv(csv_path, usecols=lambda col: col not in COLS_TO_DROP))
    else:
        for root, _, files in os.walk(subpath):
            if "listings.csv" in files:
                dfs.append(pd.read_csv(os.path.join(root, "listings.csv"), usecols=lambda col: col not in COLS_TO_DROP))
                break

if dfs:
    listings = pd.concat(dfs, ignore_index=True)
else:
    listings = pd.DataFrame()

listings.info()

In [ ]:
# REVIEW_COLS_TO_DROP = {
#     "reviewer_name", "date"
# }
# review_dfs = []

# for sub in os.listdir("./data"):
#     subpath = os.path.join("./data", sub)
#     if not os.path.isdir(subpath):
#         continue
#     csv_path = os.path.join(subpath, "reviews.csv")
#     if os.path.exists(csv_path):
#         review_dfs.append(pd.read_csv(csv_path, usecols=lambda col: col not in REVIEW_COLS_TO_DROP))
#     else:
#         for root, _, files in os.walk(subpath):
#             if "reviews.csv" in files:
#                 review_dfs.append(pd.read_csv(os.path.join(root, "reviews.csv"), usecols=lambda col: col not in REVIEW_COLS_TO_DROP))
#                 break

# if review_dfs:
#     reviews = pd.concat(review_dfs, ignore_index=True)
# else:
#     reviews = pd.DataFrame()

# reviews.info()

In [ ]:
# 1. Drop righe duplicate
dupes = listings.duplicated().sum()
listings = listings.drop_duplicates().reset_index(drop=True)
print(f"Righe duplicate rimosse: {dupes}")

In [ ]:

# 2. Drop righe con target nullo (price) o con più del 70% di valori nulli
null_price = listings["price"].isna().sum()
listings = listings.dropna(subset=["price"])
print(f"Righe con price nullo rimosse: {null_price}")

# Righe con più del 70% di valori nulli
thresh = int(0.70 * listings.shape[1])
sparse_mask = listings.isna().sum(axis=1) > thresh
sparse_count = sparse_mask.sum()
listings = listings[~sparse_mask].reset_index(drop=True)
print(f"Righe con >70% nulli rimosse: {sparse_count}")

In [ ]:
# 3. Drop righe con accommodates < 1
low_acc = (listings["accommodates"] < 1).sum()
listings = listings[listings["accommodates"] >= 1].reset_index(drop=True)
print(f"Righe con accommodates < 1 rimosse: {low_acc}")

## Analisi esplorativa


In [ ]:
listings.head()

In [ ]:
listings.info()

Il dataset contiene 44 variabili. 

La variabile da predire per il primo task è la variabile `price`, la 12-esima del DataFrame.

Applico il metodo describe a listing per avere una prima idea della distribuzione dei dati.

In [ ]:
listings.describe()

Per il task di regressione su `price` è necessario filtrare i dati per rimuovere le feature non necessarie. In particolare è necessario rimuovere le feature riguardanti il tasso di occupazione, in quanto non sono importanti per la predizione del prezzo.

Creiamo quindi un nuovo DataFrame `lst_for_price_analysis` che contiene solo le feature necessarie per la predizione del prezzo. In particolare, rimuoviamo le feature riguardanti il tasso di occupazione e la disponibilità:

- `minimum_nights`, `maximum_nights`, `minimum_minimum_nights`, `maximum_minimum_nights`, `minimum_maximum_nights`, `maximum_maximum_nights`, `minimum_nights_avg_ntm`, `maximum_nights_avg_ntm`
- `calendar_updated`, `has_availability`, `availability_30`, `availability_60`, `availability_90`, `availability_365`, `calendar_last_scraped`
- `number_of_reviews`, `number_of_reviews_ltm`, `number_of_reviews_l30d`, `availability_eoy`, `number_of_reviews_ly`
- `estimated_occupancy_l365d`, `estimated_revenue_l365d`


Manteniamo quindi solo le feature che potrebbero essere utili per la predizione del prezzo, ovvero:
- `id`
- `listing_url`
- `neighbourhood_cleansed`
- `latitude`
- `longitude`
- `property_type`
- `room_type`
- `accommodates`
- `bathrooms`
- `bedrooms`
- `beds`
- `amenities`
- `price`

In [ ]:
filtered_features = [
    "id",
    "listing_url",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "amenities",
    "price"
]

lst_for_price_analysis = listings[filtered_features]

lst_for_price_analysis.info()

Il nuovo DataFrame `lst_for_price_analysis` contiene quindi 12 variabili, di cui 11 sono feature e 1 è la variabile da predire. Le feature sono sono perlopiù numeriche. Ci sono alcune feature categoriche:
- `neighbourhood_cleansed`
- `property_type`
- `room_type`
- `amenities`

Che possono essere importanti per la predizione del prezzo, ma che vanno trattate in modo diverso dalle feature numeriche. In particolare, è necessario trasformare le feature categoriche in nuove variabili binarie.
Inoltre bisogna trattare anche il valore di `price`, che è una stringa. Per poter utilizzare questa variabile e poterla visualizzare nei plot, è necessario trasformarla in un valore numerico.

Vediamo ora il formato della variabile `price`.

In [ ]:
print(lst_for_price_analysis["price"])

Come si può vedere, la variabile `price` è una stringa che contiene il simbolo del dollaro e le virgole per le migliaia. Per poter utilizzare questa variabile e poterla visualizzare nei plot, è necessario trasformarla in un valore numerico:

In [ ]:
lst_for_price_analysis["price"] = (
    lst_for_price_analysis["price"]
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
)
lst_for_price_analysis["price"]

Adesso `price` è una variabile numerica che può essere utilizzata per la predizione del prezzo.

Ora applichiamo il metodo describe al nuovo DataFrame `lst_for_price_analysis` per avere una prima idea della distribuzione dei dati dopo il filtraggio.

In [ ]:
lst_for_price_analysis.describe()

La `latitudine`, come aspettato, è compresa tra 35.6 e 46.5, mentre la `longitudine` tra 9 e 18.5, che corrispondono alla posizione geografica dell'Italia. Per quanto riguarda le variabili `accomodates`, `bathrooms`, `bedrooms` e `beds`, si nota che sono presenti dei valori outliers.

Difatti il numero massimo di `accomodates` è 16, ancora accettabile, ma ben più alto dal valore del percentile 75 che equivale a 5. Per di più il numero massimo di `bathrooms` è 100, di `bedrooms` è 44 e di `beds` è 50, che sono valori molto elevati e potrebbero essere considerati outliers risultato di errori di immissione.

Questi valori si discostano significativamente dai dati, e possono influenzare negativamente la performance del modello di regressione. Per questo motivo, è necessario trattarli prima di procedere con la fase di modellazione.

Per quanto riguarda la variabile `price`, si nota che il prezzo massimo è 80000, mentre il prezzo del 75-esimo percentile è 168. Questo indica che ci sono dei valori di prezzo molto elevati che potrebbero anche essi essere considerati outliers.

Prima di andare a visualizzare con dei plot la distribuzione delle variabili, trattiamo i valori mancanti. Visualizziamo per ogni variabile il numero di valori mancanti:

In [ ]:
na_values = lst_for_price_analysis.isna().sum()
print(na_values)

Come possiamo vedere le variabili `bathrooms`, `bedrooms` e `beds` contengono un numero poco significativo di valori mancanti rispetto ai dati totali. Si suppone che questi valori mancanti siano dovuti a errori di immissione, siccome è impossibile avere un alloggio con 0 camere o 0 letti. Per quanto riguarda i bagni, è possibile che ci siano alloggi senza bagno, ma è più probabile che si tratti di errori di immissione. 

Per questi motivi:
- Per le variabili `bedrooms` e `beds`, si suppone che i valori mancanti siano dovuti a errori di immissione e verranno tolti dal dataset.
- Per la variabile `bathrooms`, si suppone che effettivamente ci siano alloggi senza bagno, quindi i valori mancanti verranno sostituiti con 0.

In [ ]:
lst_for_price_analysis = lst_for_price_analysis.dropna(subset=["bedrooms", "beds"])
lst_for_price_analysis["bathrooms"] = lst_for_price_analysis["bathrooms"].fillna(0)
na_values = lst_for_price_analysis.isna().sum()
print(na_values)

Ora non abbiamo più valori mancanti nelle variabili `bathrooms`, `bedrooms` e `beds`, e possiamo procedere con la visualizzazione della distribuzione delle variabili tramite dei plot.

Come primo plot, è possibile visualizzare la distribuzione della variabile `price` tramite un istogramma. In questo modo è possibile vedere se ci sono dei valori di prezzo molto elevati che potrebbero essere considerati outliers.

In [ ]:
lst_for_price_analysis["price"].plot.hist(
    bins=50,
    log=True,
)

Come si può vedere, la distribuzione della variabile `price` è molto sbilanciata, con la maggior parte dei valori concentrati tra 0 e 10000, con alcuni valori molto elevati che potrebbero essere considerati outliers. Questa cosa può essere vista anche con il boxplot:

In [ ]:
lst_for_price_analysis["price"].plot.box()

Tuttavia un numero così elevato potrebbe indicare la presenza di servizi di lusso, come ville o castelli, che potrebbero essere presenti nel dataset. Per questo motivo, è necessario analizzare più a fondo questi valori per capire se sono effettivamente outliers o se rappresentano una categoria di servizi di lusso.

Generiamo ora degli istogrammi per le variabili `accomodates`, `bathrooms`, `bedrooms` e `beds` per visualizzare meglio la presenza di outliers.

In [ ]:
cols = ["accommodates", "bathrooms", "bedrooms", "beds"]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel() # Appiattisce la matrice di assi in un array 1D per iterare più facilmente

for ax, col in zip(axes, cols):
    lst_for_price_analysis[col].dropna().plot.hist(
        bins=50,
        log=True,
        ax=ax,
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

Come si può vedere bathrooms, bedrooms e beds presentano dei valori molto elevati che sono il risultato di probabili errori di immissione. Per questo motivo questi valori saranno eliminati dal dataset:
- `bathrooms` < 20
- `bedrooms` < 22
- `beds` < 35

In [ ]:
lst_wout_outl_p_a = lst_for_price_analysis[
    (lst_for_price_analysis["bathrooms"] < 20) &
    (lst_for_price_analysis["bedrooms"] < 22) &
    (lst_for_price_analysis["beds"] < 35)
]

Adesso generiamo i boxplot per tutte le variabili per visualizzare il risulato del filtraggio degli outliers dati da errori di immissione.

In [ ]:
cols = ["accommodates", "bathrooms", "bedrooms", "beds", "price"]

fig, axes = plt.subplots(len(cols) // 2 + len(cols) % 2, len(cols) // 2, figsize=(12, 10))
axes = axes.ravel() # Appiattisce la matrice di assi in un array 1D per iterare più facilmente

for ax, col in zip(axes, cols):
    lst_wout_outl_p_a[col].dropna().plot.box(
        ax=ax,
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

Ora proviamo a visualizzare la relazione tra `price` e le variabili `accomodates`, `bathrooms`, `bedrooms` e `beds` tramite dei scatter plot. In questo modo è possibile vedere se ci sono delle relazioni tra queste variabili e il prezzo, e se ci sono dei valori di prezzo molto elevati che potrebbero essere considerati outliers.

In [ ]:
cols = ["accommodates", "bathrooms", "bedrooms", "beds"]
colors = ["blue", "orange", "green", "red"]

sample = lst_wout_outl_p_a.sample(n=5000, random_state=7112004)

fig, axes = plt.subplots(len(cols) // 2 + len(cols) % 2, len(cols) // 2, figsize=(12, 10))
axes = axes.ravel()

for ax, col in zip(axes, cols):
    sample[["price", col]].plot.scatter(
        x=col,
        y="price",
        ax=ax,
        logy= True,
        c=colors[cols.index(col)]
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Price")

plt.tight_layout()
plt.show()

In tutti e quattro i grafici si vede una tendenza positiva, anche se debole: all'aumentare del numero di `accomodates`, `bathrooms`, `bedrooms` e `beds`, aumenta anche il prezzo. Sicuramente variabili categoriche che abbiamo lasciato fuori da questa analisi, come `neighbourhood_cleansed`, `property_type`, `room_type` e `amenities`, possono avere anche esse un impatto sul prezzo, e potrebbero essere utili per migliorare la performance del modello di regressione.

Adesso analizziamo queste variabili. Estraiamo i valori unici per ogni variabile categorica:

In [ ]:
neighbourhoods = lst_for_price_analysis["neighbourhood_cleansed"].value_counts()
property_types = lst_for_price_analysis["property_type"].value_counts()
room_types = lst_for_price_analysis["room_type"].value_counts()
amenities = lst_for_price_analysis["amenities"].value_counts()

In [ ]:
print(neighbourhoods.head())
neighbourhoods.describe()

In [ ]:
print(property_types.head())
print(property_types.describe())

In [ ]:
print(room_types.head())
print(room_types.describe())

In [ ]:
print(amenities.head())
amenities.describe()

Come si può vedere, la variabile `neighbourhood_cleansed` ha 1011 valori unici, `property_type` ha 118 valori unici, `room_type` ha 4 valori unici e `amenities` ha 183716 valori unici.

`property_type` e `room_type` hanno un numero di categorie alto, quindi verranno presi i top 50 più frequenti, mentre per `room_type` che ha solo 4 categorie le visualizzeremo tutte.

per quanto riguarda `amenities`, è una variabile molto complessa, in quanto contiene una lista di amenità per record. Per questo motivo, è necessario analizzare questa variabile in modo diverso rispetto alle altre variabili categoriche. Estraiamo tutte le amenità presenti nel dataset e contiamo la frequenza di ogni amenità. In questo modo è possibile vedere quali sono le amenità più frequenti e se ci sono delle amenità che potrebbero essere utili per la predizione del prezzo.

Proviamo ora a visualizzare la distribuzione di queste variabili categoriche tramite dei grafici a barre, per capire se ci sono delle categorie molto frequenti che potrebbero essere utili per il modello di regressione.

Adesso andremo a visualizzare la distribuzione delle amenities. Per fare questo, siccome le anemities sono una stringa che rappresenta una lista di stringhe, è necessario:
1. trasformare la stringa in una lista di stringhe
2. esplodere la lista in modo da avere una riga per ogni amenità
3. contare la frequenza di ogni amenità

In questo modo è possibile ottenere un risultato simile a quello delle altre variabili categoriche, con la frequenza di ogni categoria presente nel dataset.

In [ ]:
import ast
all_amenities = lst_for_price_analysis["amenities"].apply(ast.literal_eval).explode().value_counts()
all_amenities

Anche in questo caso, avenfo 15206 ameninities uniche, si visualizzeranno solo le 50 più frequenti, per avere un'idea della distribuzione delle amenità presenti nel dataset.

In [ ]:
top_neighbourhoods = neighbourhoods.head(50)
top_neighbourhoods.plot.bar(figsize=(14, 6), title="Top 50 Neighbourhoods")

Per quanto riguarda i quartieri, le categorie più frequenti sono i centri storici delle città più grandi, in quanto sono le zone più turistiche e quindi più richieste per l'affitto di case vacanze. Oltre ai centri storici, sono presenti anche quartieri residenziali e quartieri periferici, che potrebbero essere meno richiesti ma comunque presenti nel dataset. I livelli di granularità dei quartieri sono diversi, con alcuni quartieri che rappresentano intere città, mentre altri rappresentano solo dei quartieri specifici all'interno di una città.

In [ ]:
top_property_types = property_types.head(50)
top_property_types.plot.bar(figsize=(14, 6), title="Top 50 Property Types")

Si può vedere come le categorie più frequenti di `property_type` siano concentrate su appartamenti, case vacanze e case indipendenti, che sono le tipologie di alloggio più richieste per l'affitto di case vacanze. Oltre a queste tipologie, sono presenti anche altre tipologie di alloggio meno richieste ma comunque presenti nel dataset.

In [ ]:
top_amenities = all_amenities.head(50)
top_amenities.plot.bar(figsize=(14, 6), title="Top 50 Amenities")

Si può notare come le categorie hanno una distribuzione piuttosto uniforme, con nessuna categoria che rappresenta una percentuale eccessiva rispetto alle altre. Tuttavia è da considerare che questo campo contiene circa 16000 categorie uniche, quindi è possibile che alcune categorie siano rappresentate da un numero molto basso di istanze.

Nella parte del pre-processing, è possibile decidere di mantenere solo i servizi più frequenti, in modo da ridurre la dimensionalità del dataset e migliorare la performance del modello di regressione. Un altro dato utile potrebbe essere contare il numero di amenità presenti in ogni record, in modo da avere una variabile numerica che rappresenta la quantità di servizi offerti da ogni alloggio, che potrebbe essere ulteriormente utile per la predizione del prezzo.

In [ ]:
room_types.plot.bar(log=True)

Questo grafico mostra la distribuzione delle tipologie di alloggio presenti nel dataset. Come si può vedere, le tipologie più frequenti sono gli appartamenti, seguiti dalle stanze private. Il grafico è in scala logaritmica, in quanto ci sono solo un migliario di stanze d'hotel o stanze condivise, mentre per le altre tipologie di alloggio sono presenti decine di migliaia di istanze.

Le ultime variabili da esplorare che possono essere utili per la predizione del prezzo sono le densità di letti, camere e bagni per ogni alloggio. Aggiungiamo quindi quattro nuove variabili al dataset:
- `beds_per_person` = `beds` / `accomodates`: il numero di letti per persona. Un valore più alto di questa variabile potrebbe indicare un alloggio più confortevole, e quindi potrebbe essere associato a un prezzo più elevato.
- `bedrooms_per_person` = `bedrooms` / `accomodates`: il numero di camere da letto per persona. Un valore più alto di questa variabile potrebbe indicare un alloggio più spazioso, e quindi potrebbe essere associato a un prezzo più elevato.
- `bathrooms_per_person` = `bathrooms` / `accomodates`: il numero di bagni per persona. Un valore più alto di questa variabile potrebbe indicare un alloggio più confortevole, e quindi potrebbe essere associato a un prezzo più elevato.
- `beds_per_bedroom` = `beds` / `bedrooms`: il numero di letti per camera da letto. Un valore più alto di questa variabile potrebbe indicare un alloggio più spazioso, e quindi potrebbe essere associato a un prezzo più elevato.

In [ ]:
lst_w_densities = lst_wout_outl_p_a.copy()
lst_w_densities["beds_per_person"] = lst_w_densities["beds"] / lst_w_densities["accommodates"]
lst_w_densities["bedrooms_per_person"] = lst_w_densities["bedrooms"] / lst_w_densities["accommodates"]
lst_w_densities["bathrooms_per_person"] = lst_w_densities["bathrooms"] / lst_w_densities["accommodates"]
lst_w_densities["beds_per_bedroom"] = lst_w_densities["beds"] / lst_w_densities["bedrooms"]
print(lst_w_densities)

Ora creiamo un grafico nello stile di quello superiore, per visualizzare se ci sono delle relazioni tra queste nuove variabili e il prezzo.

In [ ]:
cols = ["beds_per_person", "bedrooms_per_person", "bathrooms_per_person", "beds_per_bedroom"]
sample = lst_w_densities.sample(n=7000, random_state=7112004)

fig, axes = plt.subplots(len(cols) // 2 + len(cols) % 2, len(cols) // 2, figsize=(12, 10))
axes = axes.ravel()

for ax, col in zip(axes, cols):
    sample[["price", col]].plot.scatter(
        x=col,
        y="price",
        ax=ax,
        logy= True,
        logx= True,
        c=colors[cols.index(col)]
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Price")

plt.tight_layout()
plt.show()

Tutte e quattro le variabili non mostrano una relazione molto forte con il prezzo. Per una visualizzazione più chiara è stato utilizzato il logaritmo anche dell'asse x. Possono però essere effettuate le seguenti osservazioni:
- `beds_per_person` e `bedrooms_per_person`: si nota una lieve concentrazione dei prezzi più elevati in corrispondenza di valori prossimi a 1, ovvero listing con un letto o una camera per ospite. Tuttavia la dispersione è elevata, quindi il potere predittivo di queste feature è limitato.
- `bathrooms_per_person`: non emerge alcuna tendenza chiara. La distribuzione è sostanzialmente piatta lungo l'asse x, indicando una scarsa correlazione con il prezzo.
- `beds_per_bedroom`: si osserva una leggera tendenza inversa, infatti listing con più letti per camera (ipoteticamente dormitori o strutture condivise) tendono ad avere prezzi più bassi. Questa feature potrebbe quindi catturare indirettamente anche la tipologia di alloggio.